In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
print(os.getenv("SPARK_LOCAL_IP"))

127.0.0.1


In [2]:
from pyspark.sql import SparkSession

In [ ]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("day-3")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)


:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5f43bf32-5a82-4359-92d2-805e67276e43;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 137ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.ap

**Task 1**

Read products.csv from S3 with no options. Print the schema. 

Then read it again with inferSchema=True. What columns changed type?

In [4]:
products = spark.read.csv("s3a://pyspark-30-days-rahul-2026/data/products.csv")
products.printSchema()

26/07/26 23:54:33 WARN CredentialProviderListFactory: Credentials option fs.s3a.aws.credentials.provider contains AWS v1 SDK entry com.amazonaws.auth.profile.ProfileCredentialsProvider; mapping to software.amazon.awssdk.auth.credentials.ProfileCredentialsProvider
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)



In [5]:
products = spark.read.csv("s3a://pyspark-30-days-rahul-2026/data/products.csv", header="true", inferSchema="true")
products.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- cost_price: double (nullable = true)
 |-- supplier: string (nullable = true)
 |-- stock_quantity: integer (nullable = true)



**Task 2**

Define an explicit StructType schema for products.csv: product_id (String), product_name (String), category (String), sub_category (String), unit_price (Double), cost_price (Double), supplier (String), stock_quantity (Integer). Read the file and verify the schema.

In [6]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,DoubleType
schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("sub_category", StringType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("cost_price", DoubleType(), True),
    StructField("supplier", StringType(), True),
    StructField("stock_quantity", IntegerType(), True),
])
products1 = spark.read.csv("s3a://pyspark-30-days-rahul-2026/data/products.csv", header="true", schema=schema)
products1.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- cost_price: double (nullable = true)
 |-- supplier: string (nullable = true)
 |-- stock_quantity: integer (nullable = true)



**Task 3**

Read customers.csv from S3 using the chained .option() style with an explicit schema.

Columns: customer_id, first_name, last_name, email, city, state, country, signup_date (Date), segment.

 Show the first 5 rows.

In [7]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DateType
schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name",StringType(),True),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("signup_date(Date)",DateType(),True),
    StructField("segment",StringType(),True)
])
customers=spark.read.\
    option("header","true").\
    schema(schema).\
    csv("s3a://pyspark-30-days-rahul-2026/data/customers.csv")
customers.show(5)
customers.printSchema()

26/07/26 23:54:52 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: customer_id, first_name, last_name, email, city, state, country, signup_date, segment
 Schema: customer_id, first_name, last_name, email, city, state, country, signup_date(Date), segment
Expected: signup_date(Date) but found: signup_date
CSV file: s3a://pyspark-30-days-rahul-2026/data/customers.csv


+-----------+----------+---------+--------------------+-----------+-----+-------+-----------------+----------+
|customer_id|first_name|last_name|               email|       city|state|country|signup_date(Date)|   segment|
+-----------+----------+---------+--------------------+-----------+-----+-------+-----------------+----------+
|       C001|     James| Anderson|james.anderson@em...|   New York|   NY|    USA|       2021-03-15|Enterprise|
|       C002|     Maria|   Garcia|maria.garcia@emai...|Los Angeles|   CA|    USA|       2021-05-22|       SMB|
|       C003|    Robert|  Johnson|robert.johnson@em...|    Chicago|   IL|    USA|       2020-11-08|Enterprise|
|       C004|     Linda| Martinez|linda.martinez@em...|    Houston|   TX|    USA|       2022-01-30|       SMB|
|       C005|   Michael|    Brown|michael.brown@ema...|    Phoenix|   AZ|    USA|       2021-07-19|   Startup|
+-----------+----------+---------+--------------------+-----------+-----+-------+-----------------+----------+
o

**Task 4**

Read orders_dirty.csv using PERMISSIVE mode with _corrupt_record. 

Print total rows and bad row count. Then read it with DROPMALFORMED mode — compare the counts.

 Why does DROPMALFORMED still return 15 rows for type mismatches but drops the 3 column-count rows?

In [8]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType
)
from pyspark.sql.functions import col

# Define schema
schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("order_date", DateType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("discount_pct", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("region", StringType(), True),

    # Required for corrupt records
    StructField("_corrupt_record", StringType(), True)
])

# Read CSV in PERMISSIVE mode
permissive_df = (
    spark.read
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(schema)
    .csv("s3a://pyspark-30-days-rahul-2026/data/orders_dirty.csv")
)

# Materialize DataFrame
permissive_df.cache()
permissive_df.count()

# Count total rows
total_rows = permissive_df.count()

# Count bad rows
bad_rows = permissive_df.filter(
    col("_corrupt_record").isNotNull()
).count()

# Count good rows
good_rows = total_rows - bad_rows

print(f"Total Rows : {total_rows}")
print(f"Bad Rows   : {bad_rows}")
print(f"Good Rows  : {good_rows}")

# Show corrupt records
permissive_df.filter(
    col("_corrupt_record").isNotNull()
).show(truncate=False)

Total Rows : 15
Bad Rows   : 3
Good Rows  : 12
+--------+-----------+-------------------+----------+--------+----------+------------+------+--------------+------+----------------------------------------------------------------+
|order_id|customer_id|product_id         |order_date|quantity|unit_price|discount_pct|status|payment_method|region|_corrupt_record                                                 |
+--------+-----------+-------------------+----------+--------+----------+------------+------+--------------+------+----------------------------------------------------------------+
|O0004   |C004       |MISSING_COLUMNS_ROW|NULL      |NULL    |NULL      |NULL        |NULL  |NULL          |NULL  |O0004,C004,MISSING_COLUMNS_ROW                                  |
|O0007   |C007       |TOO                |NULL      |NULL    |NULL      |NULL        |THIS  |ROW           |HERE  |O0007,C007,TOO,MANY,EXTRA,COLUMNS,IN,THIS,ROW,HERE,EXTRA1,EXTRA2|
|O0012   |ONLY       |THREE              |NULL  

In [9]:
dropmalformed_df = (
    spark.read
    .option("header", "true")
    .option("mode", "DROPMALFORMED")
    .schema(schema)
    .csv("s3a://pyspark-30-days-rahul-2026/data/orders_dirty.csv")
)

# Cache DataFrame
dropmalformed_df.cache()

# Trigger cache
total_rows = dropmalformed_df.count()

print(f"DROPMALFORMED Total Rows : {total_rows}")

DROPMALFORMED Total Rows : 12


In [12]:
dropmalformed_df.show()

+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+---------------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount_pct|   status|payment_method| region|_corrupt_record|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+---------------+
|   O0001|       C001|      P001|2023-01-05|       2|   1299.99|        10.0|Delivered|   Credit Card|   East|           NULL|
|   O0002|       C002|      P005|2023-01-07|       1|    449.99|         0.0|Delivered|        PayPal|   West|           NULL|
|   O0003|       C003|      P003|2023-01-10|       4|    349.99|        15.0|Delivered|   Credit Card|Midwest|           NULL|
|   O0005|       C005|      P002|2023-01-15|       3|     29.99|         0.0|Delivered|   Credit Card|   West|           NULL|
|   O0006|       C006|      P008|2023-01-18|       1|    199.99|        10.0|Delivered|   Credit Card|   East| 

26/07/27 03:49:47 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 921160 ms exceeds timeout 120000 ms
26/07/27 03:49:47 WARN SparkContext: Killing executors is not supported by current scheduler.
26/07/27 03:59:45 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

DROPMALFORMED mode removes only records that Spark cannot parse structurally, such as rows having extra or missing columns.

In this dataset, the original file contained 15 records. After reading with DROPMALFORMED, Spark returned 12 records because 3 records had column-count mismatches and were dropped.

Rows with type mismatches are not always removed because Spark can still parse the row structure and replace invalid values with NULL. Therefore, DROPMALFORMED mainly handles structural errors, not every type conversion issue.